In [ ]:
# imports
import numpy as np
import scipy.linalg as slin

from dual_pfc_funcs import getParams, gen_GP

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.backends.backend_pdf import PdfPages
plt.style.use('scifigs.mplstyle')
SAVE_FIG = False

# params
color_map = getParams()['color_map']
num_neurons = 20
num_timepoints = 1000
GP_tau = 50
base_p = 1e-2
c = 5
base_add = 0.05

In [ ]:
# create figure to plot generated latents
fig,ax = plt.subplots(3,1,tight_layout=True)

# generate across-area latent
across_z = gen_GP(GP_tau, num_timepoints, seed=0)
ax[0].plot(across_z,color=color_map['across'])

# generate within-area 1 latent
within_z = gen_GP(GP_tau, num_timepoints, seed=2, N=2)
within_z1 = within_z[:,0]
ax[1].plot(within_z1,color=color_map['within1'])

# generate within-area 2 latent
within_z2 = within_z[:,1]
ax[2].plot(within_z2,color=color_map['within2'])

if SAVE_FIG:
    pdf = PdfPages('figs/sim_latents.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# generate across-area loadings
np.random.seed(0)
across_loadings1 = np.random.randn(num_neurons) + 1
across_loadings1 /= slin.norm(across_loadings1)

across_loadings2 = np.random.randn(num_neurons) + 1
across_loadings2 /= slin.norm(across_loadings2)

# generate within-area 1 loadings
np.random.seed(1)
within_loadings1 = np.random.randn(num_neurons) + 1
within_loadings1 /= slin.norm(within_loadings1)

# generate within-area 2 loadings
np.random.seed(6)
within_loadings2 = np.random.randn(num_neurons) + 1
within_loadings2 /= slin.norm(within_loadings2)

# generate spike trains
area1_raster = np.zeros((num_neurons,num_timepoints))
area2_raster = np.zeros((num_neurons,num_timepoints))

for ineuron in range(num_neurons):
    # area 1
    zi = across_loadings1[ineuron]*across_z + within_loadings1[ineuron]*within_z1
    if np.max(zi) <= 0:
        zi[zi <= 0] = base_p
    else:
        zi = zi / np.max(zi) / c # control overall firing by d
        zi = zi + base_add
    zi[zi <= 0] = base_p
    area1_raster[ineuron,:] = np.random.binomial(np.ones((num_timepoints,)).astype('int'), zi)
    
    # area 2
    zi = across_loadings2[ineuron]*across_z + within_loadings2[ineuron]*within_z2
    if np.max(zi) <= 0:
        zi[zi <= 0] = base_p
    else:
        zi = zi / np.max(zi) / c # control overall firing by d
        zi = zi + base_add
    zi[zi <= 0] = base_p
    area2_raster[ineuron,:] = np.random.binomial(np.ones((num_timepoints,)).astype('int'), zi)

In [ ]:
# plot rasters
fig,ax = plt.subplots(2,1,figsize=(4,6),sharex=True,sharey=True)

ax[0].scatter(np.where(area1_raster)[1],np.where(area1_raster)[0],marker='|',color='k',linewidth=0.3)
ax[1].scatter(np.where(area2_raster)[1],np.where(area2_raster)[0],marker='|',color='k',linewidth=0.3)

ax[0].set_title('area 1 neurons',color=color_map['within1'])
ax[0].set_xlim([0,1000])
ax[0].set_ylim([-0.5,num_neurons])
ax[0].set_xticks([])
ax[0].set_yticks([])
ax[0].spines[['left','bottom']].set_visible(False)

ax[1].set_title('area 2 neurons',color=color_map['within2'])
ax[1].set_xticks([])
ax[1].set_yticks([])
ax[1].spines[['left','bottom']].set_visible(False)

if SAVE_FIG:
    pdf = PdfPages('figs/sim_rasters.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# Define colors for the diverging colormap
across_colors = [(0, 'gray'), (0.5, 'white'), (1, color_map['across'])]
within1_colors = [(0, 'gray'), (0.5, 'white'), (1, color_map['within1'])]
within2_colors = [(0, 'gray'), (0.5, 'white'), (1, color_map['within2'])]

across_cmap = mcolors.LinearSegmentedColormap.from_list("my_diverging_cmap", across_colors)
within1_cmap = mcolors.LinearSegmentedColormap.from_list("my_diverging_cmap", within1_colors)
within2_cmap = mcolors.LinearSegmentedColormap.from_list("my_diverging_cmap", within2_colors)

# Create a figure and a set of subplots (1 row, 3 columns)
fig, axes = plt.subplots(1, 4, figsize=(15, 5)) # Adjust figsize as needed

# Plot each heatmap on its corresponding axis
im0 = axes[0].imshow(across_loadings1[:,np.newaxis], cmap=across_cmap, vmin=-1, vmax=1)
axes[0].set_title('Across-area \n co-fluctuation pattern  \n for area 1', color=color_map['across'])
axes[0].set_yticks([]),axes[0].set_xticks([])
fig.colorbar(im0, ax=axes[0], orientation='vertical')

im1 = axes[1].imshow(across_loadings2[:,np.newaxis], cmap=across_cmap, vmin=-1, vmax=1)
axes[1].set_title('Across-area \n co-fluctuation pattern \n for area 2', color=color_map['across'])
axes[1].set_yticks([]),axes[1].set_xticks([])
fig.colorbar(im1, ax=axes[1], orientation='vertical')

im2 = axes[2].imshow(within_loadings1[:,np.newaxis], cmap=within1_cmap, vmin=-1, vmax=1)
axes[2].set_title('Within-area \n co-fluctuation pattern  \n for area 1', color=color_map['within1'])
axes[2].set_yticks([]),axes[2].set_xticks([])
fig.colorbar(im2, ax=axes[2], orientation='vertical')

im3 = axes[3].imshow(within_loadings2[:,np.newaxis], cmap=within2_cmap, vmin=-1, vmax=1)
axes[3].set_title('Within-area \n co-fluctuation pattern \n for area 2', color=color_map['within2'])
axes[3].set_yticks([]),axes[3].set_xticks([])
fig.colorbar(im3, ax=axes[3], orientation='vertical')

if SAVE_FIG:
    pdf = PdfPages('figs/sim_loadings.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()